# Deep learning II: shallow neural networks

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

All the code of this lecture: the ReLU, a shallow network with one hidden unit and with three, every stage of the network as a plot, its least-squares loss on twenty training points, and a better fit by hand. There is nothing to type - run the cells in order and compare the output with the lecture.

Later cells use the variables and functions of earlier ones: Runtime > Run all runs everything at once.

Interactive companion to this lecture: [Shallow Networks](https://birkenkrahe.github.io/go/dl-shallow-demo) - move the kink of one hidden unit, set all ten parameters of the network on sliders, and watch more hidden units fit a curve.

## Code-along 1. One ReLU in code (demo)

- Load math and plotting libraries:

In [ ]:
# numerical arrays
import numpy as np
# plotting
import matplotlib.pyplot as plt

- The ReLU works on every element of an array at once:

  `np.maximum(0, preactivation)` compares every element with 0 and keeps the larger of the two. It is not `np.max`, which returns the single largest element of an array.

In [ ]:
def ReLU(preactivation):
    """Rectified linear unit: 0 for negative inputs, the input otherwise"""
    # element-wise maximum of 0 and the input
    activation = np.maximum(0, preactivation)
    return activation

# try it on five numbers
print(ReLU(np.array([-2, -0.5, 0, 0.5, 2])))

- The model with one hidden unit: y = φ₀ + φ₁ a[θ₁₀ + θ₁₁ x].

- Example: φ₀ = 0, φ₁ = 2, θ₁₀ = -1, θ₁₁ = 1, so y = 0 + 2 a[-1 + x]. For
  each x: compute the preactivation z = -1 + x, apply the ReLU, multiply
  by φ₁ = 2 and add φ₀ = 0:

| x | z = -1 + x | a[z] = max(0, z) | y = 0 + 2 a[z] |
|---:|---:|---:|---:|
| -1 | -2 | 0 | 0 |
| 0 | -1 | 0 | 0 |
| 0.5 | -0.5 | 0 | 0 |
| 1 | 0 | 0 | 0 |
| 1.5 | 0.5 | 0.5 | 1 |
| 2 | 1 | 1 | 2 |
| 3 | 2 | 2 | 4 |

- For x ≤ 1 the ReLU returns 0 and y = 0. For x > 1 it passes z on, so
  y = 2 (-1 + x) = -2 + 2x: one kink at x = 1.

- The same model in code, with the inputs of the table:

  Read the function from the inside out: `theta_10 + theta_11 * x` is the preactivation (a line), `ReLU` cuts its negative part to 0, `phi_1` scales the result, and `phi_0` is added.

In [ ]:
def shallow_1_1_1(x, phi_0, phi_1, theta_10, theta_11):
    """Shallow network: 1 input, 1 output, 1 hidden unit"""
    # line -> ReLU -> weight, plus the offset
    return phi_0 + phi_1 * ReLU(theta_10 + theta_11 * x)

# the inputs of the table
x_table = np.array([-1, 0, 0.5, 1, 1.5, 2, 3])
print(shallow_1_1_1(x_table, phi_0=0, phi_1=2, theta_10=-1, theta_11=1))

- Plot the model with the ReLU, and the same parameters without it:

  The dashed line is the same model without the ReLU. For x > 1 the two curves lie on top of each other, because the ReLU passes positive values on unchanged; for x < 1 the model stays at 0.

In [ ]:
# 100 inputs from -2 to 4
x = np.linspace(-2, 4, 100)
# with the ReLU: one kink at x = 1
plt.figure()
plt.plot(x, shallow_1_1_1(x, phi_0=0, phi_1=2, theta_10=-1, theta_11=1),
         label='with ReLU')
# without the ReLU (a[z] = z): a straight line
plt.plot(x, 0 + 2 * (-1 + 1 * x), '--', label='without ReLU')
plt.xlabel('Input, $x$'); plt.ylabel('Output, $y$')
plt.title('D = 1 shallow network: one ReLU')
plt.legend()
# show the figure
plt.show()

- Companion, card 1 (one hidden unit: move the kink, switch the ReLU
  off): https://birkenkrahe.github.io/ml_demos/shallow-network.html#relu

## Code-along 2. The network in code (demo)

- The network with one input, three hidden units and one output:

  Four steps, one line of code per hidden unit in each: preactivations, activations, weighted activations, output. The function returns the output and also the three stages, so that they can be plotted afterwards.

In [ ]:
def shallow_1_1_3(x, phi_0, phi_1, phi_2, phi_3,
                  theta_10, theta_11, theta_20, theta_21, theta_30, theta_31):
    """Shallow network: 1 input, 1 output, 3 hidden units"""
    # preactivations: three lines
    pre_1 = theta_10 + theta_11 * x
    pre_2 = theta_20 + theta_21 * x
    pre_3 = theta_30 + theta_31 * x

    # activations: pass each line through the ReLU
    act_1 = ReLU(pre_1)
    act_2 = ReLU(pre_2)
    act_3 = ReLU(pre_3)

    # weight the activations with phi_1, phi_2, phi_3
    w_act_1 = phi_1 * act_1
    w_act_2 = phi_2 * act_2
    w_act_3 = phi_3 * act_3

    # add the weighted activations and the offset phi_0
    y = phi_0 + w_act_1 + w_act_2 + w_act_3

    # the output and every stage, for the plots
    return (y, (pre_1, pre_2, pre_3), (act_1, act_2, act_3),
            (w_act_1, w_act_2, w_act_3))

- Ten parameters and 100 inputs between 0 and 1:

  The six θ are the input side (one offset and one slope per hidden unit), the four φ the output side (the offset and one weight per hidden unit). `np.arange(0, 1, 0.01)` gives 0.00, 0.01, ..., 0.99: the end value 1 is not included.

In [ ]:
# input side: three lines
theta_10 = 0.3  ; theta_11 = -1.0
theta_20 = -1.0 ; theta_21 = 2.0
theta_30 = -0.5 ; theta_31 = 0.65
# output side: offset and three weights
phi_0 = -0.3 ; phi_1 = 2.0 ; phi_2 = -1.0 ; phi_3 = 7.0

# inputs 0.00, 0.01, ..., 0.99
x = np.arange(0, 1, 0.01)

# run the network on all inputs at once
y, pre, act, w_act = shallow_1_1_3(x, phi_0, phi_1, phi_2, phi_3,
                                   theta_10, theta_11, theta_20,
                                   theta_21, theta_30, theta_31)
# one output per input
print(x.shape, y.shape)

## Code-along 3. Plot every stage (demo)

- A 3 x 3 grid: one row per stage, one column per hidden unit:

  Read the figure column by column: each column is one hidden unit on its way from a line (top) to a clipped line (middle) to a weighted clipped line (bottom).

In [ ]:
def plot_stages(x, pre, act, w_act):
    """Rows: preactivation, activation, weighted activation.
    Columns: hidden units 1, 2, 3."""
    fig, ax = plt.subplots(3, 3, figsize=(8.5, 8.5))
    fig.tight_layout(pad=3.0)
    rows = [(pre, 'Preactivation'), (act, 'Activation'),
            (w_act, 'Weighted act.')]
    # red, blue, green line for units 1, 2, 3
    colors = ['r-', 'b-', 'g-']
    for i, (stage, label) in enumerate(rows):
        for d in range(3):
            ax[i, d].plot(x, stage[d], colors[d])
            ax[i, d].set_xlim([0, 1]); ax[i, d].set_ylim([-1, 1])
        ax[i, 0].set_ylabel(label)
    # x labels on the bottom row only
    for d in range(3):
        ax[2, d].set_xlabel('Input, $x$')

In [ ]:
# the stages of the network from the last code block
plot_stages(x, pre, act, w_act)
# show the figure
plt.show()

- Each φ_d scales its unit's activation: the sign says flip or not,
  the size says stretch or shrink. The kink stays put.
  - φ_1 = 2.0: no flip, the "on" part twice as steep as act_1.
  - φ_2 = -1.0: flips act_2 upside down, same steepness.
  - φ_3 = 7.0: no flip, the "on" part seven times as steep.

## Code-along 4. The loss (demo)

- The least-squares loss, as for the line:

  $$L[\phi] = \sum_{i=1}^{I} \left(f[x_i, \phi] - y_i\right)^2$$

- Loss function and twenty training points:

  The network runs on the 20 training inputs only, because the loss compares predictions and true outputs at exactly these points. `y_predict, *_ = ...` keeps the first return value (the output) and drops the three stages.

In [ ]:
def least_squares_loss(y_train, y_predict):
    """Sum of the squared differences between true and predicted y"""
    return np.sum((y_train - y_predict) ** 2)

# twenty training points
x_train = np.array([0.09291784,0.46809093,0.93089486,0.67612654,0.73441752,
                    0.86847339,0.49873225,0.51083168,0.18343972,0.99380898,
                    0.27840809,0.38028817,0.12055708,0.56715537,0.92005746,
                    0.77072270,0.85278176,0.05315950,0.87168699,0.58858043])
y_train = np.array([-0.15934537,0.18195445,0.45127015,0.13921448,0.09366691,
                    0.30567674,0.37229117,0.40716968,-0.08131792,0.41187806,
                    0.36943738,0.3994327,0.01906257,0.35820410,0.45256496,
                    -0.0183121,0.02957665,-0.24354444,0.14803884,0.26824970])

# run the network on the training inputs only: keep y, drop the stages
y_predict, *_ = shallow_1_1_3(x_train, phi_0, phi_1, phi_2, phi_3,
                              theta_10, theta_11, theta_20,
                              theta_21, theta_30, theta_31)
loss = least_squares_loss(y_train, y_predict)
print(f'Loss = {loss:.3f}')

- The network output with the training data:

In [ ]:
def plot_output(x, y, x_data=None, y_data=None, title=None):
    """Network output y over x, optionally with data points (magenta)"""
    fig, ax = plt.subplots()
    ax.plot(x, y)
    ax.set_xlabel('Input, $x$'); ax.set_ylabel('Output, $y$')
    ax.set_xlim([0, 1]); ax.set_ylim([-1, 1])
    # 'mo' = magenta circles, no line
    if x_data is not None:
        ax.plot(x_data, y_data, 'mo')
    if title is not None:
        ax.set_title(title)

In [ ]:
# curve over all 100 inputs, training points on top
plot_output(x, y, x_train, y_train, title=f'Loss = {loss:.3f}')
# show the figure
plt.show()

## Code-along 5. Fit by hand (demo)

- The kinks are in good places, but the curve is mostly too low, and
  its first piece falls where the data rise. Change only the four φ
  (offset and weights) and keep the θ:

  What the new values do: φ₀ = 0.4 lifts the whole curve. φ₁ = -2.5 is negative, so the first piece, where only unit 1 is on, now rises towards the first joint. φ₂ = -0.7 gives the fall after x = 0.5, and φ₃ = 5.3 the rise after x = 0.77. The θ are unchanged, so the joints stay at 0.3, 0.5 and 0.77.

In [ ]:
# new output side; input side unchanged
phi_0 = 0.4 ; phi_1 = -2.5 ; phi_2 = -0.7 ; phi_3 = 5.3

# run the network again: on the training inputs and on the grid
y_predict, *_ = shallow_1_1_3(x_train, phi_0, phi_1, phi_2, phi_3,
                              theta_10, theta_11, theta_20,
                              theta_21, theta_30, theta_31)
y, pre, act, w_act = shallow_1_1_3(x, phi_0, phi_1, phi_2, phi_3,
                                   theta_10, theta_11, theta_20,
                                   theta_21, theta_30, theta_31)
loss = least_squares_loss(y_train, y_predict)
print(f'Loss = {loss:.3f}')

In [ ]:
# the new curve with the training points
plot_output(x, y, x_train, y_train, title=f'Loss = {loss:.3f}')
# show the figure
plt.show()

- From 9.385 to 0.168 by hand, with 4 of the 10 parameters. Training
  does the same for all ten at once: gradient descent on a loss
  surface with ten dimensions instead of two.

- Compare with the best line for the same 20 points: a line cannot
  follow data that rise, fall and rise again.

  `np.polyfit(x_train, y_train, 1)` fits a line by least squares and returns the slope first, then the intercept. No other line has a lower loss on these data.

In [ ]:
# least-squares line through the training points: slope, intercept
line_phi_1, line_phi_0 = np.polyfit(x_train, y_train, 1)
# the line's predictions for the training inputs
y_line = line_phi_0 + line_phi_1 * x_train
# its loss: the lowest loss any line can reach
line_loss = least_squares_loss(y_train, y_line)
print(f'Best line: y = {line_phi_0:.3f} + {line_phi_1:.3f} x')
print(f'Loss = {line_loss:.3f}')

- The best line (2 parameters) stays at 0.645; the network fitted by
  hand reaches 0.168. No search gets a line lower: the limit is the
  model, not the search.

- Companion, card 2 (all ten parameters on sliders, every stage, the
  loss): https://birkenkrahe.github.io/ml_demos/shallow-network.html#network